# ISIF solution of the RTE
## comparison of 2 quadratures to compute the temperature in a stratified grey atmosphere

 $$
   \mu\partial_z I + \kappa I = \kappa(1-a) \sigma T^4 + a\kappa\int_{-1}^1I d\mu,~~-1<\mu<1,~~0<z<Z
  $$

  $$
    I(0,\mu)|_{\mu>0} = \mu C_e \sigma T_e^4 + q_0\int_{-1}^0 I \mu d \mu,~~ I(Z,\mu)|_{\mu<0} = C_s \sigma T_s^4 \delta(\mu+\mu_s)
  $$

In [1]:
import numpy as np
import time

## Scaling Constants

In [2]:
B0, T0 = 1.4744e-8, 4798.0 # scaling constants 
stefan = (np.pi**4) / 15

Data to specify the problem 

In [3]:
Ce, Cs = 2.0, 0.2e-5    # Intensities of EM radiation from Earth and Sun
drho = -0.7  # density gradient
Zatmo = 1.2 # TAO (top of atmosphere in km)
Z = Zatmo * (1 + drho * Zatmo / 2)  # optical thickness

Te = (273 + 18) / T0 # Earth surface temperature in scaled units
Ts = 5800 / T0 # Sun apparent temperature in scaled units
kappanu = 0.5 # absorption coefficient (divided by density)
q0 = -0.3   # intensity of albeedo
mus = 0.5  # cosine of direction of collimated solar radiation


The PDE is converted into an integral equation by the method of characteristics:

$$
J_0(z) := \frac12\int_{-1}^1 I d\mu =  \frac{C_e}2 \sigma T_e^4 E_3(\kappa z)
	+\frac{c_0}2 E_2( \kappa z)
	+\frac{c_s}2 \sigma T_s^4 e^{-\frac{\kappa(Z-z)}{\mu_s}}
%	\cr&
	+  \frac{\kappa}2\int_0^Z E_1( \kappa|z'-z|)\bar J_0(z')d z'
$$

where $E_i$ is the ith exponential integral

$$
c_0 =  -q_0
[ \kappa \int_0^Z E_2(\kappa z)J_0(z)d z
	+\mu_s c_s\sigma T_s^4e^{-\frac{\kappa Z}{\mu_s}}].
$$
# ISIF computes $J_0$ for a given r.h.s., updates the r.h.s. etc

## Algorithmic constants

In [4]:
Nz = 60  # nb points in z
quadra = 1  # 0,1 quadrature precision order
kmax = 15  # nb fixed point iterations
dz = Z / (Nz - 1) #mesh size

## Exponential integrals (the last one is the first one without the log)

In [5]:
def expint_E1(t):
    """ Précision controled by Kexpint """
    t1 = abs(t)+1e-15 # strictly >0 argumeent only
    Kexpint = 10
    gaNtaua = 0.577215664901533  
    ak = t1
    soNtaue = -gaNtaua - np.log(t1) + ak
    for k in range(2, Kexpint):
        ak *= -t1 * (k - 1) / (k*k)
        soNtaue += ak
        
    return soNtaue

def expint_E2(t):
    t1=abs(t);
    return np.exp(-t1) - t1*expint_E1(t1);

def expint_E3(t):
    t1=abs(t);
    return (np.exp(-t1) - t1*expint_E2(t1))/2;

def expint_E1b(t):
    t1 = abs(t)+1e-15 
    Kexpint = 10
    gaNtaua = 0.577215664901533
    ak = t1
    soNtaue = -gaNtaua + ak # log part handled by hand
    for k in range(2, Kexpint):
        ak *= -t1 * (k - 1) / (k*k)
        soNtaue += ak
        
    return soNtaue


# The ISIF loop
## Initialization of the ISIF loop

In [6]:
z_mesh = np.linspace(0, Z, Nz)
T = np.zeros(Nz)    # Temperature array
S = np.zeros(Nz)    # Energy source array
J0 = np.zeros(Nz)   # Radiative flux array
J0log = np.zeros(Nz) # Logarithmic part of the quadrature for E1 if quadra=1
I0 = Ce*stefan*Te**4
IZ = Cs*stefan*Ts**4
# Precompute J00 and J0Z arrays vectorially
J00Z = 0.5 * I0 * expint_E3(kappanu * z_mesh) + 0.5 * IZ * np.exp(-kappanu * (Z - z_mesh)/mus) 
# Vectorized  for integrations
# Creates 2D grids of z (rows) and zp (columns) to eliminate internal loops
z_grid, zp_grid = np.meshgrid(z_mesh, z_mesh, indexing="ij")


global T, S, J0
# Initialize T and S
T[:] = Te/2
S[:] = stefan * T**4
J0 = J00Z
    
start_time = time.time()
    
# ISIF loop
for k in range(kmax):
    Q0 = -q0*kappanu*mus*IZ*np.exp(-kappanu*Z/mus)  - q0*kappanu*dz*np.dot(expint_E2(kappanu * z_mesh),J0)
        
    diff_z = zp_grid - z_grid
    J0z = J00Z + 0.5*Q0*expint_E2(kappanu*z_mesh)
    # quadrature control
    if quadra == 0:
        J0 = J0z  +  0.5*dz*np.dot(expint_E1(kappanu *(-dz/2+diff_z)), S)
    elif quadra == 1:
        # the log part of E_1 is treated separately 
        J0log = (diff_z) * (np.log(kappanu * ( 1e-10+ np.abs(diff_z))) - 1) - (diff_z - dz) * (np.log(kappanu * ( 1e-10+ np.abs(diff_z-dz))) - 1)
        J0 = J0z +  0.5*dz*np.dot(expint_E1b(kappanu *(-dz/2+diff_z)), S) - 0.5 * np.dot(J0log, S)

    # Update Temperature via Fixed Point
    T = (J0/stefan)**0.25

     #  updates to S
    S[0]=0
    for i in range(1,Nz):
        S[i] = kappanu * (J0[i]+J0[i-1])/2
    
    # print some values during ISIF loop
    print(f"T(0)= {T[0]*T0-273:.4f} at iteration {k}")


T(0)= -12.8000 at iteration 0
T(0)= -0.7114 at iteration 1
T(0)= 3.5617 at iteration 2
T(0)= 5.1559 at iteration 3
T(0)= 5.7652 at iteration 4
T(0)= 6.0000 at iteration 5
T(0)= 6.0909 at iteration 6
T(0)= 6.1260 at iteration 7
T(0)= 6.1396 at iteration 8
T(0)= 6.1449 at iteration 9
T(0)= 6.1469 at iteration 10
T(0)= 6.1477 at iteration 11
T(0)= 6.1480 at iteration 12
T(0)= 6.1482 at iteration 13
T(0)= 6.1482 at iteration 14


## Print some values during the ISIF loop and at the end

In [7]:
print("Altitude  T  J0")
for i in range(Nz):
        print(f"{(np.sqrt(1+2*drho*z_mesh[i])-1)/drho :.4f}   {T[i]*T0-273  :.4f}   {J0[i]/B0:.4f}")


Altitude  T  J0
-0.0000   6.1482   5046.5122
0.0118   6.6308   5081.5034
0.0238   6.8627   5098.3780
0.0358   7.0043   5108.7064
0.0480   7.0893   5114.9092
0.0603   7.1332   5118.1174
0.0726   7.1451   5118.9873
0.0851   7.1309   5117.9463
0.0977   7.0946   5115.2938
0.1104   7.0392   5111.2501
0.1233   6.9670   5105.9837
0.1363   6.8798   5099.6269
0.1494   6.7791   5092.2860
0.1626   6.6658   5084.0478
0.1760   6.5411   5074.9840
0.1895   6.4057   5065.1550
0.2032   6.2601   5054.6115
0.2170   6.1051   5043.3968
0.2310   5.9410   5031.5478
0.2452   5.7683   5019.0963
0.2595   5.5872   5006.0692
0.2740   5.3981   4992.4901
0.2887   5.2012   4978.3788
0.3036   4.9966   4963.7523
0.3187   4.7846   4948.6249
0.3339   4.5652   4933.0087
0.3495   4.3385   4916.9131
0.3652   4.1046   4900.3460
0.3812   3.8635   4883.3131
0.3974   3.6152   4865.8183
0.4138   3.3596   4847.8638
0.4306   3.0968   4829.4498
0.4476   2.8267   4810.5751
0.4650   2.5491   4791.2366
0.4826   2.2638   4771.4294
0.5